# التحليل المقارن لمعماريات EfficientNet — D-MorphNet

هذا الدفتر ينفّذ قسم **"Comparative Analysis of EfficientNet Architectures"**: مقارنة ثلاثة إصدارات بإعداداتها المذكورة حرفياً في الجدول ٢، لفهم سبب اختيار EfficientNet-B6.

## الجدول ٢ — القيم المنشورة في الورقة (المرجع)

| النموذج | الاستخراج | المصنّف | بُعد السمات | الدقة % | Precision | Recall | F1 |
|---|---|---|---|---|---|---|---|
| EfficientNet-B0 | CNN أساسي | **Softmax** | 1280 | 62.4 | 0.63 | 0.61 | 0.62 |
| EfficientNet-B5 | سمات عميقة | **SVM خطي** | 2048 | 66.13 | 0.66 | 0.66 | 0.66 |
| EfficientNet-B6 (المقترح) | سمات عميقة | **SVM (RBF)** | 2304 | **89.9** | **0.90** | **0.90** | **0.90** |

## تفسير النتائج (كما في الورقة)
- **B0**: معمارية ضحلة بقدرة محدودة على التقاط الأنماط الوجهية الدقيقة — جيد للتعرف الأساسي على الوجوه لكنه يعجز عن رصد الآثار البصرية الصغيرة في صور المورف.
- **B5**: مع SVM كمستخرج سمات تحسّن الأداء — يلتقط تفاصيل وجهية أدق تساعد المصنّف على التفريق.
- **B6**: الأعمق والأكثر معاملات — يلتقط أدق التفاصيل، ومع SVM (نواة RBF) يعطي أفضل النتائج.

كما نرسم **منحنى ROC (الشكل ٣)**: في الورقة بلغت **AUC = 0.965**، ويرتفع المنحنى بحدة قرب نقطة البداية — أي أن النظام يكتشف معظم صور المورف مع إبقاء الإنذارات الخاطئة منخفضة.

> ⚙️ فعّل GPU قبل التشغيل — سنستخرج سمات B0 وB5 من الصور مباشرة.


## ١) تجهيز البيئة وقيم الورقة المرجعية


In [ ]:
!pip -q install scikit-learn joblib pandas

import os, glob, random, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# القيم المنشورة في الجدول 2 من الورقة
PAPER_TABLE2 = pd.DataFrame({
    "Model": ["EfficientNet-B0", "EfficientNet-B5", "EfficientNet-B6 (Proposed)"],
    "Classifier": ["Softmax", "SVM (Linear)", "SVM (RBF)"],
    "Feature Dimension": [1280, 2048, 2304],
    "Accuracy (%)": [62.4, 66.13, 89.9],
    "Precision": [0.63, 0.66, 0.90],
    "Recall": [0.61, 0.66, 0.90],
    "F1-Score": [0.62, 0.66, 0.90],
}).set_index("Model")

PAPER_AUC = 0.965
gpus = tf.config.list_physical_devices('GPU')
print("GPU:", gpus if gpus else "⚠️ لا يوجد GPU — فعّله من إعدادات الجلسة!")
PAPER_TABLE2


## ٢) تحميل الصور المعالجة وسمات B6 الجاهزة

- الصور المعالجة (528×528) لاستخراج سمات B0 وB5 منها مباشرة.
- سمات B6 المحفوظة مسبقاً (`.npz`) من دفتر التمثيل العميق — لا نعيد استخراجها.


In [ ]:
PROC_DIR = "/content/DMorphNet_processed"
FEAT_DIR = "/content/DMorphNet_features"
DRIVE_DIR = "/content/drive/MyDrive/DMorphNet_models"


def find_file(name):
    for root in [FEAT_DIR, DRIVE_DIR]:
        p = os.path.join(root, name)
        if os.path.exists(p):
            return p
    return None


need_drive = (not os.path.isdir(os.path.join(PROC_DIR, "train"))) or \
             (find_file("effb6_ft_train.npz") is None and
              find_file("effb6_train.npz") is None)
if need_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir(os.path.join(PROC_DIR, "train")):
        shutil.unpack_archive("/content/drive/MyDrive/DMorphNet_processed.zip",
                              PROC_DIR)

LABEL_MAP = {"real": 0, "morph": 1}


def list_files(split):
    paths, labels = [], []
    for cls, lab in LABEL_MAP.items():
        fs = sorted(glob.glob(os.path.join(PROC_DIR, split, cls, "*.jpg")))
        paths += fs
        labels += [lab] * len(fs)
    return paths, np.array(labels)


# سمات B6 الجاهزة (الأولوية لما بعد الضبط الدقيق)
PREFIX = "effb6_ft" if find_file("effb6_ft_train.npz") else "effb6"
b6 = {}
for split in ["train", "test"]:
    data = np.load(find_file(f"{PREFIX}_{split}.npz"))
    b6[split] = data["X"].astype(np.float32)

tr_paths, tr_y = list_files("train")
te_paths, te_y = list_files("test")
print(f"تدريب: {len(tr_paths)} صورة — اختبار: {len(te_paths)} صورة")
print(f"سمات B6: تدريب {b6['train'].shape} / اختبار {b6['test'].shape}")


## ٣) الإعدادات ودوال الاستخراج والتقييم المشتركة

- للمقارنة العادلة: **نفس عيّنة التدريب** (6,000 صورة) و**كامل مجموعة الاختبار** (2,000 صورة) لكل النماذج الثلاثة. *(ضع `SUBSET = None` لاستخدام كامل التدريب — أبطأ بكثير.)*
- أحجام الإدخال الأصلية لكل إصدار: B0 ← 224، B5 ← 456، B6 ← 528.
- دالة تقييم موحّدة تحسب الدقة وPrecision/Recall/F1 وقيم ROC.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc)
from tensorflow.keras.applications.efficientnet import preprocess_input

AUTOTUNE = tf.data.AUTOTUNE
SUBSET = 6000              # حجم عيّنة التدريب للمقارنة (None = الكل)

rng = np.random.RandomState(SEED)
tr_idx = rng.permutation(len(tr_paths))[:SUBSET] if SUBSET else np.arange(len(tr_paths))
sub_tr_paths = [tr_paths[i] for i in tr_idx]
y_tr01, y_te01 = tr_y[tr_idx], te_y                    # صيغة 0/1
y_tr = np.where(y_tr01 == 0, -1, +1)                   # صيغة -1/+1 لـ SVM
y_te = np.where(y_te01 == 0, -1, +1)


def extract_features(model_cls, size, paths):
    # استخراج سمات مجمّدة بأي إصدار EfficientNet
    base = model_cls(include_top=False, weights="imagenet",
                     pooling="avg", input_shape=(size, size, 3))

    def load(p):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.image.resize(tf.cast(img, tf.float32), [size, size])
        return preprocess_input(img)

    ds = (tf.data.Dataset.from_tensor_slices(list(paths))
          .map(load, num_parallel_calls=AUTOTUNE).batch(32).prefetch(AUTOTUNE))
    return base.predict(ds, verbose=1)


def compute_metrics(y_true_pm, pred_pm, scores):
    # المقاييس الأربعة + نقاط منحنى ROC
    fpr, tpr, _ = roc_curve(y_true_pm, scores, pos_label=+1)
    return {
        "Accuracy (%)": accuracy_score(y_true_pm, pred_pm) * 100,
        "Precision": precision_score(y_true_pm, pred_pm, pos_label=+1),
        "Recall": recall_score(y_true_pm, pred_pm, pos_label=+1),
        "F1-Score": f1_score(y_true_pm, pred_pm, pos_label=+1),
        "AUC": auc(fpr, tpr),
        "_roc": (fpr, tpr),
    }

results = {}
print("✅ الإعدادات جاهزة")


## ٤) التجربة ١ — EfficientNet-B0 الأساسي + Softmax

كما في الجدول ٢: B0 يُستخدم كـ **CNN أساسي بمصنّف Softmax** (وليس SVM):
- عمود فقري B0 مجمّد (سمات 1280) + طبقة Softmax تُدرّب على بياناتنا.
- درجة ROC = احتمال فئة "مدموجة" من Softmax.


In [ ]:
from tensorflow.keras.applications import EfficientNetB0

print("استخراج سمات B0 (1280 بُعداً) ...")
b0_tr = extract_features(EfficientNetB0, 224, sub_tr_paths)
b0_te = extract_features(EfficientNetB0, 224, te_paths)

# رأس Softmax على السمات المجمّدة (المكافئ العملي لـ CNN أساسي + Softmax)
sc0 = StandardScaler().fit(b0_tr)
head_b0 = tf.keras.Sequential([
    tf.keras.layers.Input((b0_tr.shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(2, activation="softmax"),
])
head_b0.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                loss="sparse_categorical_crossentropy", metrics=["accuracy"])
head_b0.fit(sc0.transform(b0_tr), y_tr01, epochs=10, batch_size=256, verbose=2)

prob_b0 = head_b0.predict(sc0.transform(b0_te), verbose=0)[:, 1]
pred_b0 = np.where(prob_b0 >= 0.5, +1, -1)
results["EfficientNet-B0"] = compute_metrics(y_te, pred_b0, prob_b0)
results["EfficientNet-B0"]["Classifier"] = "Softmax"
results["EfficientNet-B0"]["Feature Dimension"] = b0_tr.shape[1]

print(f"\nB0 + Softmax → الدقة = {results['EfficientNet-B0']['Accuracy (%)']:.2f}%"
      f"  (الورقة: 62.4%)")


## ٥) التجربة ٢ — EfficientNet-B5 + SVM خطي

كما في الجدول ٢: سمات B5 العميقة (2048 بُعداً، إدخال 456×456) مع **SVM خطي**.


In [ ]:
from tensorflow.keras.applications import EfficientNetB5

print("استخراج سمات B5 (2048 بُعداً) ...")
b5_tr = extract_features(EfficientNetB5, 456, sub_tr_paths)
b5_te = extract_features(EfficientNetB5, 456, te_paths)

sc5 = StandardScaler().fit(b5_tr)
svm_b5 = LinearSVC(C=1.0, random_state=SEED)
svm_b5.fit(sc5.transform(b5_tr), y_tr)

pred_b5 = svm_b5.predict(sc5.transform(b5_te))
score_b5 = svm_b5.decision_function(sc5.transform(b5_te))
results["EfficientNet-B5"] = compute_metrics(y_te, pred_b5, score_b5)
results["EfficientNet-B5"]["Classifier"] = "SVM (Linear)"
results["EfficientNet-B5"]["Feature Dimension"] = b5_tr.shape[1]

print(f"\nB5 + SVM خطي → الدقة = {results['EfficientNet-B5']['Accuracy (%)']:.2f}%"
      f"  (الورقة: 66.13%)")


## ٦) التجربة ٣ — EfficientNet-B6 المقترح + SVM بنواة RBF

كما في الجدول ٢: سمات B6 (2304 بُعداً) مع **SVM بنواة RBF** — النواة اللاخطية ترسم حداً فاصلاً أكثر مرونة في فضاء السمات، وهي المستخدمة في النظام المقترح.


In [ ]:
b6_tr, b6_te = b6["train"][tr_idx], b6["test"]

sc6 = StandardScaler().fit(b6_tr)
svm_b6 = SVC(kernel="rbf", C=10.0, gamma="scale", random_state=SEED)
print(f"تدريب SVM (RBF) على {len(b6_tr)} عيّنة ...")
svm_b6.fit(sc6.transform(b6_tr), y_tr)

pred_b6 = svm_b6.predict(sc6.transform(b6_te))
score_b6 = svm_b6.decision_function(sc6.transform(b6_te))
results["EfficientNet-B6 (Proposed)"] = compute_metrics(y_te, pred_b6, score_b6)
results["EfficientNet-B6 (Proposed)"]["Classifier"] = "SVM (RBF)"
results["EfficientNet-B6 (Proposed)"]["Feature Dimension"] = b6_tr.shape[1]

print(f"\nB6 + SVM (RBF) → الدقة = "
      f"{results['EfficientNet-B6 (Proposed)']['Accuracy (%)']:.2f}%"
      f"  (الورقة: 89.9%)")


## ٧) بناء الجدول ٢: نتائجنا مقابل الورقة

نجمع نتائج التجارب الثلاث في جدول بنفس أعمدة الجدول ٢، جنباً إلى جنب مع القيم المنشورة.


In [ ]:
our_table = pd.DataFrame({
    name: {
        "Classifier": r["Classifier"],
        "Feature Dimension": r["Feature Dimension"],
        "Accuracy (%)": round(r["Accuracy (%)"], 2),
        "Precision": round(r["Precision"], 2),
        "Recall": round(r["Recall"], 2),
        "F1-Score": round(r["F1-Score"], 2),
        "AUC": round(r["AUC"], 4),
    } for name, r in results.items()
}).T

print("═══ الجدول ٢ — نتائجنا ═══")
display(our_table)
print("\n═══ الجدول ٢ — القيم المنشورة في الورقة ═══")
display(PAPER_TABLE2)

best = our_table["Accuracy (%)"].astype(float).idxmax()
print(f"\n🏆 الأفضل في تجربتنا: {best} — "
      "يؤكد استنتاج الورقة أن B6 الأعمق يلتقط الآثار الدقيقة للمورف أفضل")


## ٨) الشكل ٣ — منحنيات ROC للنماذج الثلاثة

نرسم منحنى ROC لكل نموذج على نفس الشكل:
- في الورقة بلغت AUC للنموذج المقترح **0.965**.
- لاحظ **الارتفاع الحاد قرب نقطة البداية** لمنحنى B6 — يعني اكتشاف معظم صور المورف مع إبقاء معدل الإنذارات الخاطئة منخفضاً.


In [ ]:
plt.figure(figsize=(8, 7))
styles = {"EfficientNet-B0": ":", "EfficientNet-B5": "--",
          "EfficientNet-B6 (Proposed)": "-"}
for name, r in results.items():
    fpr, tpr = r["_roc"]
    plt.plot(fpr, tpr, styles[name], linewidth=2,
             label=f"{name} (AUC = {r['AUC']:.4f})")
plt.plot([0, 1], [0, 1], color="gray", linewidth=1,
         label="تخمين عشوائي (AUC = 0.5)")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC Curves — Comparative Analysis (الشكل ٣)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

auc_b6 = results["EfficientNet-B6 (Proposed)"]["AUC"]
print(f"AUC للنموذج المقترح: {auc_b6:.4f}  (الورقة: {PAPER_AUC})")
print("قيمة قريبة من 1.0 = تمييز ممتاز بين الوجوه الحقيقية والمدموجة")


## ٩) رسم مقارنة الدقة وحفظ النتائج

مخطط أعمدة يقارن دقتنا مع الدقة المنشورة لكل نموذج، ثم حفظ الجدول والنتائج في Google Drive.


In [ ]:
models = list(results.keys())
ours = [results[m]["Accuracy (%)"] for m in models]
paper_vals = list(PAPER_TABLE2["Accuracy (%)"])

x = np.arange(len(models))
plt.figure(figsize=(10, 5))
plt.bar(x - 0.2, ours, width=0.4, label="نتيجتنا")
plt.bar(x + 0.2, paper_vals, width=0.4, label="الورقة")
plt.xticks(x, ["B0\n(Softmax)", "B5\n(SVM خطي)", "B6\n(SVM RBF)"])
plt.ylabel("الدقة (%)")
plt.title("مقارنة دقة معماريات EfficientNet — نتيجتنا مقابل الورقة")
plt.legend()
plt.grid(axis="y", alpha=0.3)
for i, (o, p) in enumerate(zip(ours, paper_vals)):
    plt.text(i - 0.2, o + 0.5, f"{o:.1f}", ha="center", fontsize=9)
    plt.text(i + 0.2, p + 0.5, f"{p:.1f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

# حفظ النتائج
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = "/content/drive/MyDrive/DMorphNet_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
our_table.to_csv(os.path.join(RESULTS_DIR, "table2_comparative_analysis.csv"))
print("تم الحفظ في:", os.path.join(RESULTS_DIR, "table2_comparative_analysis.csv"))
print("\n🎉 اكتمل التحليل المقارن لمعماريات EfficientNet")
